# 13. Orchestration agentique du test-time scaling

**Pont entre NB-12 (les quatre moteurs) et NB-04 (Function Calling) / NB-09 (Production).**

Le notebook **`12_Test_Time_Scaling.ipynb`** implemente from scratch, en Python pur, quatre
moteurs d'inference : **Best-of-N**, **Tree-of-Thoughts**, **Reflexion**, et un **routeur
adaptatif** *hand-tuned* (regles de difficulte -> strategie, code en dur).

Ce notebook **NB-13** pousse l'integration une etape plus loin, au sens du projet de reference
[Iterative-Contextual-Refinements (ICR)](https://github.com/ryoiki-tokuien/Iterative-Contextual-Refinements) :
le **4eme mode d'ICR, "Agentic"**, remplace le routeur code-en-dur par un **agent planificateur**
qui, via l'**API de function calling** (pont NB-04), choisit lui-meme quel moteur invoquer
**pour chaque sous-tache**, en justifiant son choix.

> Idee centrale : plutot qu'un `if difficulte >= 4: best_of_n()` ecrit a la main, on donne a
> un LLM un **ensemble d'outils** (les moteurs) et on le laisse **raisonner** sur lequel
> appliquer. C'est exactement le saut de "code qui appelle un modele" a "modele qui appelle du code".

**Plan :**
1. Rappel compact des trois moteurs (renvoye a NB-12 pour le detail).
2. Definition des **outils** (schemas JSON pour le function calling).
3. La **boucle agentique** : le routeur raisonne, choisit un outil, l'execute, observe.
4. Demonstration sur trois types de taches (raisonnement, echantillonnage, combinatoire).
5. Pont production (NB-09) : garde-fou de cout, fallback, retris.

## 0. Setup — meme infrastructure que NB-12

On reutilise le client **OpenRouter** et le chargeur `.env` robuste de NB-12
(pour que Papermill trouve le `.env` quel que soit le `cwd`). Aucune dependance
au-dela de `openai` + `python-dotenv`.

In [1]:
# Installation des dependances (identique NB-12)
%pip install -q openai python-dotenv

import os, re, json
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# --- Chargeur .env robuste (Papermill change le cwd) ---
_env_path = None
_current = Path.cwd()
for _i in range(10):
    if (_current / ".env").exists():
        _env_path = _current / ".env"
        break
    if _current.name in ("GenAI", "MyIA.AI.Notebooks"):
        break
    _current = _current.parent
if _env_path is None:
    for _cand in (Path.cwd() / "MyIA.AI.Notebooks" / "GenAI" / ".env",
                  Path.cwd() / "GenAI" / ".env"):
        if _cand.exists():
            _env_path = _cand
            break
if _env_path is not None:
    load_dotenv(_env_path)
    print(f".env charge depuis : {_env_path}")
else:
    print("ATTENTION : aucun .env trouve.")

# --- Modeles (etat de l'art 2026, identique NB-12) ---
FAST_MODEL = os.getenv("OPENAI_MODEL_FAST", "meta-llama/llama-3.3-70b-instruct")
BIG_MODEL = os.getenv("OPENAI_MODEL_BIG", "openai/gpt-5-nano")
BATCH_MODE = os.getenv("BATCH_MODE", "true").lower() in ("1", "true", "yes")

# --- Client OpenRouter ---
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)
print(f"FAST_MODEL = {FAST_MODEL}")
print(f"BIG_MODEL  = {BIG_MODEL}")
print(f"BATCH_MODE = {BATCH_MODE}")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


.env charge depuis : C:\dev\CoursIA-2\MyIA.AI.Notebooks\GenAI\.env


FAST_MODEL = meta-llama/llama-3.3-70b-instruct
BIG_MODEL  = openai/gpt-5-nano
BATCH_MODE = False


In [2]:
def chat(prompt, system=None, model=FAST_MODEL, temperature=0.0, max_tokens=2000,
         tools=None, tool_choice=None):
    """Appel unique au LLM. Renvoie l'objet message complet (pour lire les tool_calls)
    ou le contenu texte. Renvoie None en cas d'erreur (BATCH_MODE safe)."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt}) if not isinstance(prompt, list) else messages.extend(prompt)
    try:
        resp = client.chat.completions.create(
            model=model, messages=messages, temperature=temperature,
            max_tokens=max_tokens, tools=tools, tool_choice=tool_choice,
        )
        return resp.choices[0].message
    except Exception as exc:
        print(f"  [chat] erreur ({model}) : {exc}")
        return None

# Test de connectivite (1 appel economique)
_ping = chat("Reponds uniquement par : OK", max_tokens=10)
print("Ping :", repr((_ping.content or "")[:40]) if _ping else "ECHEC")

Ping : 'OK'


## 1. Les trois moteurs, en rappel compact

On redefinit ici les **memes moteurs** que NB-12, en version condensee. Le but n'est pas
de re-expliquer ToT ou Reflexion (voir NB-12) mais de les exposer comme des **fonctions
appelables** que l'agent pourra declarer comme outils. Chaque moteur resout un probleme de
generation de code Python (spec + tests) et renvoie `(passes, total, tokens)` — sauf ToT
qui resout le **24-game** combinatoire.

In [3]:
# --- Utilitaires partages (extraire/executer du code genere) ---
def extraire_code(reponse):
    """Extrait le 1er bloc ```python ... ``` d'une reponse LLM."""
    m = re.search(r"```(?:python)?\s*(.*?)```", reponse or "", re.DOTALL)
    return m.group(1).strip() if m else (reponse or "").strip()

def executer_tests(code, probleme, timeout=5):
    """Execute `code` dans un namespace isole, puis evalue chaque test du probleme.
    Renvoie (nb_tests_passes, nb_tests_total). Echec = exception ou False."""
    total = len(probleme["tests"])
    if not code.strip():
        return 0, total
    ns = {}
    try:
        exec(code, ns)
    except Exception:
        return 0, total
    passes = 0
    for test in probleme["tests"]:
        try:
            if eval(test, ns):
                passes += 1
        except Exception:
            pass
    return passes, total

# Un mini-banque de problemes (spec + tests) pour les demos. Voir NB-12 pour la banque complete.
PROBLEMES = [
    {"id": "palindrome", "spec":
     "Ecris une fonction python `plus_long_palindrome(s)` qui renvoie le plus long palindrome "
     "contigu dans s. Renvoie '' si s vide.",
     "tests": ["plus_long_palindrome('babad') in ('bab','aba')",
               "plus_long_palindrome('racecar')=='racecar'",
               "plus_long_palindrome('')==''"]},
    {"id": "fizzbuzz", "spec":
     "Ecris une fonction python `fizzbuzz(n)` renvoyant une liste de longueur n ou l'element i "
     "(1-indexe) vaut 'Fizz' si i multiple de 3, 'Buzz' si multiple de 5, 'FizzBuzz' si multiple "
     "des deux, sinon l'entier i.",
     "tests": ["fizzbuzz(5)==[1,2,'Fizz',4,'Buzz']", "fizzbuzz(15)[-1]=='FizzBuzz'"]},
]

def _gen_code(probleme, model=FAST_MODEL, temperature=0.0, max_tokens=2000):
    prompt = probleme["spec"] + "\n\nReponds UNIQUEMENT avec le code Python dans un bloc ```python```."
    resp = chat(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    txt = resp.content if resp else ""
    return extraire_code(txt), len(txt.split())

print("Utilitaires + banque de problemes charges.")

Utilitaires + banque de problemes charges.


In [4]:
def moteur_best_of_n(probleme, model=FAST_MODEL, n=3, temperature=0.8):
    """Best-of-N : genere n solutions (temperature haute), garde celle qui passe le plus
    de tests. Robuste aux erreurs systematiques d'echantillonnage. Pont NB-12."""
    meilleur, total, tokens = 0, len(probleme["tests"]), 0
    for _ in range(n):
        code, tok = _gen_code(probleme, model=model, temperature=temperature)
        tokens += tok
        passes, _ = executer_tests(code, probleme)
        meilleur = max(meilleur, passes)
    return {"moteur": "best_of_n", "passes": meilleur, "total": total, "tokens": tokens}

def moteur_reflexion(probleme, model=FAST_MODEL, iterations=3):
    """Reflexion : boucle generateur -> execution -> critique -> regeneration.
    La memoire = la trace des tests echoues. Pont NB-12."""
    total = len(probleme["tests"])
    meilleur, tokens, feedback, code = 0, 0, "", ""
    for _ in range(iterations):
        if feedback:
            prompt = (probleme["spec"] + "\n\nCode precedent ECHEC :\n```python\n" + code +
                      "\n```\nTests echouant :\n" + feedback +
                      "\nCorrige. Reponds UNIQUEMENT avec le code dans ```python```.")
        else:
            prompt = probleme["spec"] + "\n\nReponds UNIQUEMENT avec le code dans ```python```."
        resp = chat(prompt, model=model, max_tokens=2000)
        txt = resp.content if resp else ""
        tokens += len(txt.split())
        code = extraire_code(txt)
        passes, _ = executer_tests(code, probleme)
        meilleur = max(meilleur, passes)
        if passes >= total:
            break
        # memoire : quels tests echouent
        ns = {}
        try: exec(code, ns)
        except Exception: ns = {}
        fails = []
        for t in probleme["tests"]:
            try:
                if not eval(t, ns): fails.append(t)
            except Exception as e: fails.append(f"{t}  # -> {type(e).__name__}")
        feedback = "\n".join(fails) if fails else ""
    return {"moteur": "reflexion", "passes": meilleur, "total": total, "tokens": tokens}

from itertools import combinations
def moteur_tot_24(nombres, largeur=3):
    """Tree-of-Thoughts (BFS faisceau) sur le 24-game. Domaine combinatoire ou le
    single-shot echoue systematiquement. Pont NB-12."""
    def atteignable(xs, cible=24.0, tol=1e-6):
        if len(xs) == 1: return abs(xs[0] - cible) < tol
        for i, j in combinations(range(len(xs)), 2):
            a, b = xs[i], xs[j]
            rest = [xs[k] for k in range(len(xs)) if k not in (i, j)]
            for v in (a+b, a-b, b-a, a*b):
                if atteignable(rest + [v], cible, tol): return True
            if abs(b) > tol and atteignable(rest + [a/b], cible, tol): return True
            if abs(a) > tol and atteignable(rest + [b/a], cible, tol): return True
        return False
    trouve = atteignable(list(nombres))
    return {"moteur": "tot_24", "solution_trouvee": trouve, "nombres": list(nombres)}

print("Trois moteurs charges : best_of_n, reflexion, tot_24 (rappels compacts de NB-12).")

Trois moteurs charges : best_of_n, reflexion, tot_24 (rappels compacts de NB-12).


## 2. Les outils — schemas JSON pour le function calling

Le function calling (pont **NB-04**) consiste a declarer des fonctions que le modele peut
**decider d'invoquer**. On expose ici nos trois moteurs comme trois outils. Le schema JSON
decrit les parametres ; le modele produit un `tool_call` qu'on execute cote client.

C'est le decalage cle : NB-12 **appelait** les moteurs depuis du code ; ici c'est le **modele**
qui choisit lequel appeler.

In [5]:
OUTILS = [
    {
        "type": "function",
        "function": {
            "name": "resoudre_best_of_n",
            "description": "Resout un probleme de code Python par Best-of-N : genere n solutions "
                           "et garde la meilleure. Utile quand le single-shot echoue par variance "
                           "d'echantillonnage (pas par meconnaissance). Coute N fois un appel.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id_probleme": {"type": "string", "enum": [p["id"] for p in PROBLEMES],
                                    "description": "Identifiant du probleme dans la banque."},
                    "n": {"type": "integer", "default": 3, "minimum": 1, "maximum": 5},
                },
                "required": ["id_probleme"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "resoudre_reflexion",
            "description": "Resout un probleme de code Python par Reflexion : boucle "
                           "generation-critique-regeneration avec memoire des tests echoues. "
                           "Utile quand l'erreur est systematique (le modele se trompe pareil).",
            "parameters": {
                "type": "object",
                "properties": {
                    "id_probleme": {"type": "string", "enum": [p["id"] for p in PROBLEMES]},
                    "iterations": {"type": "integer", "default": 3, "minimum": 1, "maximum": 4},
                },
                "required": ["id_probleme"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "resoudre_tot_24",
            "description": "Resout un probleme de 24-game (atteindre 24 avec 4 nombres et +,-,*,/) "
                           "par Tree-of-Thoughts (recherche combinatoire). A utiliser UNIQUEMENT "
                           "pour le 24-game, pas pour du code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombres": {"type": "array", "items": {"type": "number"},
                                "description": "Les 4 nombres du 24-game."},
                },
                "required": ["nombres"],
            },
        },
    },
]

# Table de dispatch : nom d'outil -> fonction Python a executer.
def _probleme_par_id(pid):
    return next((p for p in PROBLEMES if p["id"] == pid), None)

def executer_outil(nom, args):
    """Execute l'outil demande par le modele et renvoie un resultat serialisable."""
    if nom == "resoudre_best_of_n":
        pb = _probleme_par_id(args["id_probleme"])
        return moteur_best_of_n(pb, n=int(args.get("n", 3)))
    if nom == "resoudre_reflexion":
        pb = _probleme_par_id(args["id_probleme"])
        return moteur_reflexion(pb, iterations=int(args.get("iterations", 3)))
    if nom == "resoudre_tot_24":
        return moteur_tot_24(args["nombres"])
    return {"erreur": f"outil inconnu : {nom}"}

print(f"{len(OUTILS)} outils declares pour le function calling.")

3 outils declares pour le function calling.


## 3. La boucle agentique — le coeur de NB-13

L'agent recoit une **tache libre** (en langage naturel), raisonne, emet un `tool_call`,
on l'execute, on renvoie le resultat, et l'agent decide de s'arreter ou de reessayer avec
un autre moteur. C'est la traduction directe du mode **"Agentic"** d'ICR.

La boucle est bornee (garde-fou anti-boucle-infini, pont **NB-09** production).

In [6]:
SYSTEME_AGENT = (
    "Tu es un agent planificateur specialise dans le test-time scaling. Tu as trois outils : "
    "resoudre_best_of_n, resoudre_reflexion, resoudre_tot_24.\n"
    "Regle de choix :\n"
    "- 24-game (atteindre 24 avec 4 nombres) -> resoudre_tot_24 (UNIQUEMENT).\n"
    "- Probleme de code ou l'erreur est systematique / conceptuelle -> resoudre_reflexion.\n"
    "- Probleme de code ou le single-shot echoue par variance -> resoudre_best_of_n.\n"
    "Tu dois invoquer UN outil, puis analyser son resultat. Si la solution est trouvee "
    "(passes==total, ou solution_trouvee==True), tu t'arrettes. Sinon, tu peux essayer un autre "
    "outil une seule fois. Sois concis."
)

def agent_routeur(tache, model=BIG_MODEL, max_tours=4, max_tokens=1500, retries_message_vide=2):
    """Boucle agentique : l'agent choisit un moteur via tool-calling, l'execute, observe.
    Renvoie la trace des tours. Bornee a max_tours (anti-boucle).

    retries_message_vide : un appel de routing via OpenRouter peut sporadiquement renvoyer un
    message **vide** (ni tool_call ni contenu). Consacrer ce vide comme reponse_finale serait un
    defaut de fidelite (audit #3164 — grain Demo 1 : la 1ere re-exec a produit un message vide en
    tour 1, affichant une demo "Solution trouvee" vide). On re-tente donc le tour courant tant que
    le message est vide, avant d'abandonner. Un message final REEL non vide (ou un tool_call)
    arrete/continue normalement."""
    conversation = [{"role": "system", "content": SYSTEME_AGENT},
                    {"role": "user", "content": tache}]
    trace = []
    for tour in range(1, max_tours + 1):
        msg = None
        for _essai in range(retries_message_vide + 1):
            msg = chat(conversation, model=model, max_tokens=max_tokens, tools=OUTILS)
            # Message vide = ni outil ni texte (flakiness OpenRouter) -> on retente le tour.
            if msg is None or msg.tool_calls or (msg.content and msg.content.strip()):
                break
        if msg is None:
            trace.append({"tour": tour, "erreur": "echec appel LLM"})
            break
        if not msg.tool_calls:
            # Message final reel (non vide) : on garde une reponse finale suffisamment large pour
            # ne pas tronquer le code genere en plein milieu d'une fonction (audit #3164 — grain
            # Demo 2/3 : le slice [:200] coupait les fonctions palindrome/fizzbuzz).
            trace.append({"tour": tour, "reponse_finale": (msg.content or "")[:600]})
            break
        # L'agent a demande un (ou des) outil(s) : on execute et on nourrit la conversation
        conversation.append(msg)
        for tc in msg.tool_calls:
            nom = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except Exception:
                args = {}
            resultat = executer_outil(nom, args)
            trace.append({"tour": tour, "outil": nom, "args": args, "resultat": resultat})
            conversation.append({
                "role": "tool", "tool_call_id": tc.id,
                "content": json.dumps(resultat, ensure_ascii=False),
            })
    return trace

print("agent_routeur defini (boucle tool-calling bornee + retry message vide, pont NB-04 + NB-09).")

agent_routeur defini (boucle tool-calling bornee + retry message vide, pont NB-04 + NB-09).


## 4. Demonstration — l'agent choisit le bon moteur

On soumet trois taches de natures differentes et on observe **le choix de l'agent**.
L'interet pedagogique : le routeur hand-tuned de NB-12 etait code en dur ; ici le choix
emerge du raisonnement du modele.

In [7]:
# Demo 1 : un 24-game -> l'agent DOIT choisir ToT (les autres outils sont inadaptés).
print("=" * 64)
print("DEMO 1 — tache combinatoire (24-game avec 4,7,8,8)")
print("=" * 64)
trace1 = agent_routeur("Resous le 24-game avec les nombres [4, 7, 8, 8].")
for etape in trace1:
    print(etape)

DEMO 1 — tache combinatoire (24-game avec 4,7,8,8)


{'tour': 1, 'outil': 'resoudre_tot_24', 'args': {'nombres': [4, 7, 8, 8]}, 'resultat': {'moteur': 'tot_24', 'solution_trouvee': True, 'nombres': [4, 7, 8, 8]}}
{'tour': 2, 'reponse_finale': 'Résolution trouvée: 8 * 7 - 4 * 8 = 24.'}


In [8]:
# Demo 2 : un probleme de code -> l'agent choisit best_of_n ou reflexion selon son diagnostic.
print("=" * 64)
print("DEMO 2 — tache de generation de code (palindrome)")
print("=" * 64)
trace2 = agent_routeur("Resous le probleme de code identifie 'palindrome' de la banque.")
for etape in trace2:
    print(etape)

DEMO 2 — tache de generation de code (palindrome)


{'tour': 1, 'outil': 'resoudre_reflexion', 'args': {'id_probleme': 'palindrome', 'iterations': 3}, 'resultat': {'moteur': 'reflexion', 'passes': 3, 'total': 3, 'tokens': 121}}
{'tour': 2, 'reponse_finale': 'Résultat: résolu via resoudre_reflexion (solution trouvée).\n\nProposition de solution Python robuste (ignore les caractères non alphanumériques et la casse):\n\nTwo-pointer (efficace, sans créer une nouvelle chaîne):\ndef is_palindrome(s: str) -> bool:\n    i, j = 0, len(s) - 1\n    while i < j:\n        while i < j and not s[i].isalnum():\n            i += 1\n        while i < j and not s[j].isalnum():\n            j -= 1\n        if s[i].lower() != s[j].lower():\n            return False\n        i += 1\n        j -= 1\n    return True\n\nUtilisation simple:\n- Pour tester une seule chaîne:\nprint(is_palindrom'}


In [9]:
# Demo 3 : un autre probleme de code (fizzbuzz).
print("=" * 64)
print("DEMO 3 — tache de generation de code (fizzbuzz)")
print("=" * 64)
trace3 = agent_routeur("Resous le probleme de code identifie 'fizzbuzz' de la banque.")
for etape in trace3:
    print(etape)

DEMO 3 — tache de generation de code (fizzbuzz)


{'tour': 1, 'outil': 'resoudre_reflexion', 'args': {'id_probleme': 'fizzbuzz', 'iterations': 3}, 'resultat': {'moteur': 'reflexion', 'passes': 2, 'total': 2, 'tokens': 84}}
{'tour': 2, 'reponse_finale': 'SolutionFizzBuzz identifiée par le module de réflexion. Voici une solution Python robuste qui respecte le standard fizzbuzz.\n\nCode proposé:\ndef fizzbuzz(n):\n    res = []\n    for i in range(1, n + 1):\n        s = ""\n        if i % 3 == 0:\n            s += "Fizz"\n        if i % 5 == 0:\n            s += "Buzz"\n        res.append(s or str(i))\n    return res\n\nExemple d’utilisation:\nfor x in fizzbuzz(15):\n    print(x)\n\nCette version renvoie une liste des sorties pour les nombres 1 à n, affichant Fizz pour les multiples de 3, Buzz pour les multiples de 5, et FizzBuzz pour les multiples des deux. Sino'}


## 5. Pont vers la production (NB-09)

En production, une boucle agentique aveugle est dangereuse : cout non borne, boucles,
appels inutiles. Trois garde-fous classiques (pont **NB-09 Production Patterns**) :

- **Budget de cout** : plafond de tokens/appels au-dela duquel on bascule sur un moteur
  cheap par defaut.
- **Fallback** : si l'agent n'invoque aucun outil (hallucination), on force un moteur safe.
- **Idempotence / retris bornes** : `max_tours` deja present ; on ajoute un timeout global.

L'exercice 2 ci-dessous te fait coder le garde-fou de budget.

In [10]:
# Smoke test du garde-fou anti-boucle (max_tours).
# Un appel agent_routeur nominal converge en 2 tours (1 appel d'outil + 1 reponse finale).
# Si on borne la meme tache a max_tours=1, la boucle DOIT s'arreter des le 1er tour, avant
# que l'agent n'ait pu emettre sa reponse finale : c'est l'arret premature impose par la
# borne, exactement le mecanisme anti-boucle-infini qu'on veut demontrer.
_trace_nominal = agent_routeur("Resous le 24-game avec les nombres [1, 3, 4, 6].", max_tours=4)
_trace_borne = agent_routeur("Resous le 24-game avec les nombres [1, 3, 4, 6].", max_tours=1)
print(f"Tours nominaux (max_tours=4) : {len(_trace_nominal)}  <- chemin nominal (outil puis reponse)")
print(f"Tours bornes   (max_tours=1) : {len(_trace_borne)}  <- la borne arrete apres l'outil, avant la reponse")
_a_arret_borne = len(_trace_borne) == 1 and any("outil" in e for e in _trace_borne) and not any("reponse_finale" in e for e in _trace_borne)
print(f"max_tours=1 a tronque la boucle avant la reponse finale -> garde-fou OK : {_a_arret_borne}")

Tours nominaux (max_tours=4) : 2  <- chemin nominal (outil puis reponse)
Tours bornes   (max_tours=1) : 1  <- la borne arrete apres l'outil, avant la reponse
max_tours=1 a tronque la boucle avant la reponse finale -> garde-fou OK : True


## 6. Travaux pratiques

Les exercices sont a completer (convention C.1 : pas d'erreur volontaire, le notebook
tourne de bout en bout meme non complete). Chaque `# TODO etudiant` indique l'etape.

### Exercice 1 : ajouter un outil de self-consistency

La **self-consistency** (Wang et al. 2022) = Best-of-N + **vote a la majorite** sur la
reponse finale (pas sur les tests). Ajoute un 4eme outil `resoudre_self_consistency` a
l'agent. Pont NB-12.

**Etapes :**
1. Definis le schema JSON de l'outil (inspire-toi de `resoudre_best_of_n`, avec un
   parametre `vote_sur` decrivant ce qu'on met a la majorite).
2. Implemente la fonction Python associee dans `executer_outil`.
3. Ajoute l'outil a la liste `OUTILS` et au `SYSTEME_AGENT`.

**Indice :** tu peux reutiliser moteur_best_of_n puis appliquer un Counter sur une cle
extraite de chaque solution genere.

In [11]:
def ajouter_outil_self_consistency():
    # TODO etudiant : construis le dict du schema d'outil (cf OUTILS), implemente la
    # branche correspondante dans executer_outil, puis ajoute l'outil a OUTILS.
    schema = None
    return schema

_s = ajouter_outil_self_consistency()
print(f"Exercice 1 - self-consistency : {'schema defini' if _s else 'a completer'}")

Exercice 1 - self-consistency : a completer


### Exercice 2 : routeur avec garde-fou de budget (pont NB-09)

En production, une boucle agentique doit etre **budgetee**. Enrichis `agent_routeur`
d'un plafond de tokens au-dela duquel on stoppe.

**Etapes :**
1. Somme les tokens consommes par chaque outil execute (champ `tokens` des resultats).
2. Si le budget est atteint, arrete la boucle et renvoie une trace marquee
   `budget_epuise` (n'appelle plus l'API).
3. Garde `max_tours` comme seconde borne.

**Indice :** accumule les tokens dans une variable ; teste le seuil AVANT le prochain tour.

In [12]:
def agent_routeur_avec_budget(tache, budget_tokens=6000, model=BIG_MODEL, max_tours=4):
    # TODO etudiant : adapte la boucle de agent_routeur avec le budget cumule de tokens.
    trace = None
    return trace

_t = agent_routeur_avec_budget("Resous 'palindrome'.", budget_tokens=4000)
print(f"Exercice 2 - routeur budgete : {'trace renvoyee' if _t is not None else 'a completer'}")

Exercice 2 - routeur budgete : a completer


### Exercice 3 (avance) : orchestration multi-outils

Au lieu d'un seul moteur par tache, autorise l'agent a **chainer** : par exemple
best_of_n pour generer, puis reflexion pour affiner si best_of_n n'a pas passe tous
les tests.

**Etapes :**
1. Authorise l'agent a appeler plusieurs outils successifs dans la meme boucle.
2. Detecte l'echec partiel (`passes < total`) et laisse l'agent decider d'un 2eme outil.
3. Retourne une trace qui montre le chainage (moteur A -> echec partiel -> moteur B).

**Indice :** le system prompt doit explicitement autoriser le chainage ; max_tours doit
etre augmente.

In [13]:
def agent_multi_outils(tache, model=BIG_MODEL, max_tours=6):
    # TODO etudiant : orchestration multi-outils avec chainage et detection d'echec partiel.
    trace = None
    return trace

_m = agent_multi_outils("Resous 'fizzbuzz'.")
print(f"Exercice 3 - multi-outils : {'trace renvoyee' if _m is not None else 'a completer'}")

Exercice 3 - multi-outils : a completer


## 7. Conclusion et suite

On a remplace le routeur **hand-tuned** de NB-12 (regles codees en dur) par un **agent
planificateur** qui choisit le moteur via le **function calling** — le 4eme mode d'ICR
("Agentic"). C'est le saut conceptuel de "du code qui pilote un modele" a "un modele qui
pilote du code".

**Limites honnetes (G.2) :**
- Sur les 3 demos, l'agent et le routeur hand-tuned font le **meme choix** qualitatif (ToT
  pour le 24-game, Reflexion pour le code) : sur ce petit echantillon homogene, le routeur
  code-en-dur suffit. La valeur de l'agent apparait sur des **taches heterogenes a grande
  echelle** ou ecrire les regles a la main devient intenable. Nous n'avons **pas** mesure
  ici de comparaison quantitative cote-a-cote (taux de succes, cout en tokens) entre les
  deux : ce serait l'objet d'une etude dediee.
- Le cout en appels : mecaniquement, l'agent ajoute **un appel de planification LLM par
  tour** (le `chat(...)` qui decide de l'outil) que le routeur hand-tuned n'effectue pas.
  Sur une charge massive, ce surcout d'orchestration s'accumule (d'ou le garde-fou de
  budget de l'exercice 2).
- Le raisonnement du planificateur est lui-meme sujet au bruit : un modele faible peut
  mal etiqueter la tache. D'ou l'importance du garde-fou de budget (exercice 2).

**Suite de l'epic #2926 :**
- **Phase 2** — memoire persistante : brancher la `feedback` de Reflexion sur un vector store
  (pont NB-05 RAG) pour que les leçons survivent entre sessions.
- **Phase 3** — ToT sur de vrais problemes de recherche (cryptarithmetic, CSP) ou single-shot
  echoue systematiquement (pont series Search/Sudoku).
- **Phase 4** — courbes de scaling Snell 2024 (quand echantillonner large vs chercher profond).

**References :** ICR repo ; Snell et al. 2024 ; Yao et al. 2023 (ToT) ; Wang et al. 2022
(Self-Consistency) ; Shinn et al. 2023 (Reflexion). Anchors : NB-04, NB-05, NB-09, NB-12.